<a href="https://colab.research.google.com/github/fischdakid/is-4487-labs/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [6]:
import pandas as pd, numpy as np, time
from sklearn.model_selection import train_test_split

# Load data
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
print(df.dtypes.value_counts())
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])

# Drop leakage columns (reservation_status reveals the outcome) and high-cardinality IDs
df = df.drop(columns=['reservation_status', 'reservation_status_date', 'agent', 'company', 'country'], errors='ignore')

# X / y and 70/30 split BEFORE any preprocessing
y = df['is_canceled']
X = df.drop(columns=['is_canceled'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_test = X_train.copy(), X_test.copy()

# Impute missing values using training data only
num_cols = X_train.select_dtypes(include='number').columns
cat_cols = X_train.select_dtypes(exclude='number').columns
med = X_train[num_cols].median()
X_train[num_cols] = X_train[num_cols].fillna(med)
X_test[num_cols] = X_test[num_cols].fillna(med)
X_train[cat_cols] = X_train[cat_cols].fillna('Unknown')
X_test[cat_cols] = X_test[cat_cols].fillna('Unknown')

# One-hot encode categoricals; align test columns to train
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True).astype(float)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True).reindex(columns=X_train.columns, fill_value=0).astype(float)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Cancel rate:", round(y.mean(), 3))

Shape: (9404, 32)
int64      17
object     12
float64     3
Name: count, dtype: int64
Missing values:
 country     266
agent      2335
company    8393
dtype: int64
Train: (6582, 61) Test: (2822, 61)
Cancel rate: 0.248


### ✍️ Your Response: 🔧
1. This dataset has 9404 rows and 32 columns.

2. The dataset is full of mostly numerical data. 17 interger and 3 float columnns. The other 12 of of the columns are categorical like the hotel, meal, and market segment.

3. To clean and prepare the data I dropped reservation status and its date since it gives away the outcome. Along with this I dropped agent, company, and country because I don't see as much value from them. I ended up splitting the 70/30 first and filling numeric gaps with the training median. Lastly, I one hot encoded the categories.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [7]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

t = time.time()
nb = GaussianNB().fit(X_train, y_train)
nb_time = time.time() - t
nb_pred = nb.predict(X_test)

print(classification_report(y_test, nb_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, nb_pred))

              precision    recall  f1-score   support

           0       0.94      0.48      0.64      2121
           1       0.37      0.91      0.52       701

    accuracy                           0.59      2822
   macro avg       0.65      0.69      0.58      2822
weighted avg       0.80      0.59      0.61      2822

Confusion matrix:
 [[1021 1100]
 [  66  635]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. This model doesn't perform well, it only has a 59% accuracy. it catches 91% of cancellations but also falsely flags 1,100 bookings that actually showed up. Accuracy is misleading since 25% of booking actually cancel so using F1 or recall is better for the cancel class.

2. This model could actually be useful since its trains almost instantly making it decent for a quick screen. It could be solid for a real time alert on possible cancellations. It is not great for staffing ort overbooking decisions.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Fit scaler on training data only
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

t = time.time()
svm = SVC(kernel='linear').fit(X_train_s, y_train)
svm_time = time.time() - t
svm_pred = svm.predict(X_test_s)

print(classification_report(y_test, svm_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, svm_pred))

              precision    recall  f1-score   support

           0       0.84      1.00      0.91      2121
           1       0.98      0.41      0.58       701

    accuracy                           0.85      2822
   macro avg       0.91      0.70      0.74      2822
weighted avg       0.87      0.85      0.83      2822

Confusion matrix:
 [[2114    7]
 [ 412  289]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. This model performs better with an 85% accuracy and 98% precision on cancellations, meaning that when it flags one it has a high likelyhood of being right. The downfall here is that it only catches 41% of them whichs shows the tradeoff better than accuracy does.

2. SVM is helpful when patterns span many features at once and false alarm is costly. An example si when deciding which bookings justify a target overbooking, where you'd prefer to be sure before acting.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [9]:
from sklearn.neural_network import MLPClassifier

# True validation split carved out of the training data (test set untouched)
X_tr, X_val, y_tr, y_val = train_test_split(X_train_s, y_train, test_size=0.2, random_state=42, stratify=y_train)

t = time.time()
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42).fit(X_tr, y_tr)
nn_time = time.time() - t

print("Validation accuracy:", round(mlp.score(X_val, y_val), 4))
nn_pred = mlp.predict(X_test_s)
print(classification_report(y_test, nn_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, nn_pred))

Validation accuracy: 0.8755
              precision    recall  f1-score   support

           0       0.91      0.91      0.91      2121
           1       0.73      0.71      0.72       701

    accuracy                           0.86      2822
   macro avg       0.82      0.81      0.82      2822
weighted avg       0.86      0.86      0.86      2822

Confusion matrix:
 [[1936  185]
 [ 201  500]]


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. Neaural network is the best of the three: 86% accuracy and a 0.72 F1 that gives balanced precision and recall. Its 87.5% validation accurcy matches the test result meaning it isn't overfitting.

2. I think management would be hesitant because the model can't easily explain why a booking is flagged. In terms of forecasting staffing and inventory, results matter more than explanations so this model would be acceptible assuming they monitor it and pair it with simple reports on what drives cancellations.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [10]:
from sklearn.metrics import f1_score
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM (linear)', 'Neural Network'],
    'Accuracy': [accuracy_score(y_test, p) for p in [nb_pred, svm_pred, nn_pred]],
    'F1 (canceled)': [f1_score(y_test, p) for p in [nb_pred, svm_pred, nn_pred]],
    'Train time (s)': [nb_time, svm_time, nn_time]
}).round(4)
print(results.sort_values('Accuracy', ascending=False).to_string(index=False))
print("\nBest model by accuracy:", results.loc[results['Accuracy'].idxmax(), 'Model'])

         Model  Accuracy  F1 (canceled)  Train time (s)
Neural Network    0.8632         0.7215          3.9167
  SVM (linear)    0.8515         0.5797          3.4623
   Naive Bayes    0.5868         0.5213          0.0092

Best model by accuracy: Neural Network


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. Neural Network provides the best overall accuracy and F1. Naive Bayes is the fastest to train and the easiest to use. SVM is the simplest to interpret but misses most cancellations. Training times for SVM and Neural Network are similar but better models than Naive Bayes.

2. I would go with Neural Network. It does the best job at catching cancellations than the others with less false alarms. This is exactly what planning is looking for. Retraining the model regularly and checking it against how bookings run would also help overtime.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. My final recommendation is the Neural Network model since it is the most accurate at 86% and caught 71% of cancellations with less false alarms. This helps the hotel forecast actual occupancy, staff scheduling, inventory management, and helps set smarter overbooking limits. The only risk is that its a Black box so staff need to retrain it regularly since the booking patterns change throughout the year. Additional data that would be helpful would be customer history, price changes, booking channel info, and event data.

2. One of my leanring outcomes is related to turning models into insights for a non-technical manager to use. This assignemnt practices exactly that. I picked F1 instead of raw accuracy making the results less misleading then put them in simple business terms for the non-technical people to be able to work with.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [12]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 326955 bytes to assignment_12_bayes_svm_neural.html
